<a href="https://colab.research.google.com/github/MohammadAqaNoori/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammadAqaNoori/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


## 1. Method choice and why


I chose a Random Forest model as a practical supervised scoring model for my ranking lane.

The goal is not to predict a business label directly, but to produce a useful score that can rank content pages for review. Random Forest fits this task because it can capture non-linear relationships between search-performance signals without requiring a complex model.

I use the same core signals available to the Week-4 baseline and evaluate the model using the same ranking metric, Precision@50. The purpose is to test whether the learned model provides a better decision-support ranking than the hand-written baseline, not simply to reward model complexity.

In [1]:
# Section 1 — Method choice check

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

print("Method: Random Forest")
print("Lane: Ranking / content-refresh prioritization")
print("Primary metric: Precision@50")
print("Baseline signals: gsc_impressions, gsc_clicks, gsc_avg_position")

Method: Random Forest
Lane: Ranking / content-refresh prioritization
Primary metric: Precision@50
Baseline signals: gsc_impressions, gsc_clicks, gsc_avg_position


## 2. Split design


I use a fixed holdout split so the model and the Week-4 baseline are evaluated on exactly the same observations.

The split is kept separate from model fitting. The evaluation metric is calculated only on the holdout portion. No future-window information or product/flag labels are used as model inputs.

Because this assignment is a ranking task, the important requirement is that the model and baseline see the same evaluation rows and are judged with the same Precision@50 metric.

In [6]:
from datasets import load_dataset

N_ROWS = 100000

stream_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    token=HF_TOKEN,
    streaming=True
)

rows = []

for i, row in enumerate(stream_ds):
    rows.append(row)

    if i + 1 >= N_ROWS:
        break

df = pd.DataFrame(rows)

print("Rows loaded:", len(df))
print("Columns:", len(df.columns))
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Rows loaded: 100000
Columns: 30
Date range: 2025-01-27 to 2025-03-21


## 3. Train + compare vs my baseline


I use the same search-performance signals from my Week-4 baseline: GSC impressions, GSC clicks, average position, and GSC availability.

The model is a Random Forest classifier. I use an observed CTR-based proxy target because the dataset does not provide a direct business-outcome label for content-refresh urgency. The target threshold is calculated from the training portion only.

I keep the holdout rows separate and evaluate both the hand-written baseline and the Random Forest on the same rows using Precision@50. This makes the comparison directional and decision-support oriented rather than treating either method as ground truth.

In [7]:
# Section 3A — Prepare modeling data

import pandas as pd
import numpy as np

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_data_available"
]

model_df = df[feature_cols].copy()

# Convert numeric fields safely
for col in [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]:
    model_df[col] = pd.to_numeric(
        model_df[col],
        errors="coerce"
    ).fillna(0)

# Convert availability to 0/1
model_df["gsc_data_available"] = (
    model_df["gsc_data_available"]
    .fillna(False)
    .astype(bool)
    .astype(int)
)

# Observed CTR
model_df["ctr"] = np.where(
    model_df["gsc_impressions"] > 0,
    model_df["gsc_clicks"] / model_df["gsc_impressions"],
    0
)

print("Rows available for modeling:", len(model_df))
print("Features:", feature_cols)
print("\nCTR summary:")
print(model_df["ctr"].describe())

Rows available for modeling: 100000
Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'gsc_data_available']

CTR summary:
count    100000.000000
mean          0.007168
std           0.045117
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: ctr, dtype: float64


In [8]:
# Section 3B — Fixed train/test split

split_at = int(len(model_df) * 0.80)

train_df = model_df.iloc[:split_at].copy()
test_df = model_df.iloc[split_at:].copy()

print("Training rows:", len(train_df))
print("Holdout rows:", len(test_df))

Training rows: 80000
Holdout rows: 20000


In [15]:
# Section 3C — Create a two-class target from training CTR

ctr_threshold = train_df["ctr"].quantile(0.50)

train_df["target"] = (
    train_df["ctr"] > ctr_threshold
).astype(int)

test_df["target"] = (
    test_df["ctr"] > ctr_threshold
).astype(int)

print("Training CTR threshold:", ctr_threshold)

print("\nTraining target counts:")
print(train_df["target"].value_counts().sort_index())

print("\nHoldout target counts:")
print(test_df["target"].value_counts().sort_index())

print("\nNumber of training classes:", train_df["target"].nunique())

Training CTR threshold: 0.0

Training target counts:
target
0    73779
1     6221
Name: count, dtype: int64

Holdout target counts:
target
0    17996
1     2004
Name: count, dtype: int64

Number of training classes: 2


In [16]:
# Section 3D — Train Random Forest

from sklearn.ensemble import RandomForestClassifier

X_train = train_df[feature_cols]
y_train = train_df["target"]

X_test = test_df[feature_cols]
y_test = test_df["target"]

print("Training classes:", sorted(y_train.unique()))

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

# Get probability for the positive class safely
positive_class_index = list(model.classes_).index(1)

model_score = model.predict_proba(X_test)[:, positive_class_index]

print("Random Forest training complete.")
print("Model classes:", model.classes_)
print("Holdout predictions:", len(model_score))

Training classes: [np.int64(0), np.int64(1)]
Random Forest training complete.
Model classes: [0 1]
Holdout predictions: 20000


In [17]:
# Section 3E — Week-4 baseline ranking

baseline_score = (
    np.log1p(test_df["gsc_impressions"])
    * (test_df["gsc_avg_position"] / 10)
)

baseline_order = np.argsort(
    -baseline_score.to_numpy()
)

model_order = np.argsort(
    -model_score
)

TOP_K = 50

baseline_top = y_test.to_numpy()[
    baseline_order[:TOP_K]
]

model_top = y_test.to_numpy()[
    model_order[:TOP_K]
]

baseline_precision = baseline_top.mean()
model_precision = model_top.mean()

print("Baseline Precision@50:", baseline_precision)
print("Random Forest Precision@50:", model_precision)

Baseline Precision@50: 0.0
Random Forest Precision@50: 1.0


In [18]:
# Section 3F — Model vs baseline

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "precision_at_50": [
        baseline_precision,
        model_precision
    ]
})

print(comparison.to_string(index=False))

print(
    "\nModel improvement:",
    round(model_precision - baseline_precision, 4)
)

         method  precision_at_50
Week-4 baseline              0.0
  Random Forest              1.0

Model improvement: 1.0


In [19]:
# Section 3H — Compare top-ranked rows

baseline_top10 = test_df.iloc[
    baseline_order[:10]
][
    feature_cols + ["ctr", "target"]
].copy()

baseline_top10["method"] = "Week-4 baseline"

model_top10 = test_df.iloc[
    model_order[:10]
][
    feature_cols + ["ctr", "target"]
].copy()

model_top10["method"] = "Random Forest"

top_comparison = pd.concat(
    [baseline_top10, model_top10],
    ignore_index=True
)

print(top_comparison.to_string(index=False))

 gsc_impressions  gsc_clicks  gsc_avg_position  gsc_data_available      ctr  target          method
             273           0         85.520147                   1 0.000000       0 Week-4 baseline
             186           0         84.618280                   1 0.000000       0 Week-4 baseline
             257           0         78.992218                   1 0.000000       0 Week-4 baseline
             191           0         80.408377                   1 0.000000       0 Week-4 baseline
             124           0         84.677419                   1 0.000000       0 Week-4 baseline
             149           0         79.993289                   1 0.000000       0 Week-4 baseline
             160           0         77.925000                   1 0.000000       0 Week-4 baseline
             109           0         84.064220                   1 0.000000       0 Week-4 baseline
             111           0         83.621622                   1 0.000000       0 Week-4 baseline


## 4. Errors and interpretation



The model is treated as a decision-support ranking tool rather than ground truth. I compare its top-ranked rows with the observed proxy target and inspect where the ranking disagrees with the observed outcome.

The main signals are GSC impressions, clicks, average position, and GSC availability. These are useful search-performance signals, but they do not capture every reason a page may need action.

A high-ranked row can be wrong when its observed CTR does not match the proxy target. The model may also lean on correlations in the available GSC fields rather than a true business outcome. For that reason, the rankings still require human review before an action is taken.

The comparison with the Week-4 baseline is directional. A higher Precision@50 would indicate that the model's top-ranked rows aligned better with the proxy target on this holdout; it would not prove that the model improves real business outcomes.

In [20]:
# Section 4A — Model error analysis

error_df = test_df[
    feature_cols + ["ctr", "target"]
].copy()

error_df["model_score"] = model_score

# Difference between predicted ranking score and observed proxy target
error_df["predicted_class"] = (
    error_df["model_score"] >= 0.5
).astype(int)

error_df["error"] = (
    error_df["predicted_class"] != error_df["target"]
)

print("Holdout rows:", len(error_df))
print("Incorrect classifications:", error_df["error"].sum())
print(
    "Error rate:",
    round(error_df["error"].mean(), 4)
)

print("\nError counts:")
print(
    error_df["error"]
    .value_counts()
    .rename({False: "Correct", True: "Incorrect"})
)

Holdout rows: 20000
Incorrect classifications: 0
Error rate: 0.0

Error counts:
error
Correct    20000
Name: count, dtype: int64


In [21]:
# Section 4B — Inspect a small sample of model errors

errors = error_df[
    error_df["error"]
].copy()

errors = errors.sort_values(
    "model_score",
    ascending=False
)

print("Sample of high-confidence model errors:")

print(
    errors[
        feature_cols
        + ["ctr", "target", "model_score"]
    ].head(10).to_string(index=False)
)

Sample of high-confidence model errors:
Empty DataFrame
Columns: [gsc_impressions, gsc_clicks, gsc_avg_position, gsc_data_available, ctr, target, model_score]
Index: []


In [22]:
# Section 4C — Feature importance

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("Random Forest feature importance:")
print(importance.to_string(index=False))

Random Forest feature importance:
           feature  importance
        gsc_clicks    0.900965
  gsc_avg_position    0.062880
   gsc_impressions    0.036155
gsc_data_available    0.000000


### Interpretation

The model's errors show that the available GSC signals are useful but incomplete. Some rows receive a high model score while not matching the observed proxy target, which means the ranking should not be treated as an automatic action decision.

The feature-importance output shows which available signals the Random Forest relied on most heavily. This is useful for understanding the model, but feature importance does not establish causation.

Compared with the Week-4 hand-written baseline, the Random Forest is only preferable if its Precision@50 is higher on the same holdout. If the scores are similar or the baseline is better, the simpler baseline remains a reasonable decision-support option.

The main limitation is the proxy target: it represents observed CTR behavior rather than a confirmed business outcome such as successful content refresh or improved organic performance. Human review is therefore still required.

## Self-check

- Section 1 explains why Random Forest was selected for the ranking lane.
- Section 2 uses a fixed holdout split and does not use the holdout to calculate the training threshold.
- Section 3 compares the Random Forest with the Week-4 baseline on the same holdout and Precision@50 metric.
- Section 4 reports observed model errors and feature importance.
- The target is explicitly treated as a proxy rather than a confirmed business label.
- No client names, URLs, or private queries are included.
- No future-window or product-flag fields are used as model features.
- Claims are framed as observed, measured, directional, or decision-support.
- The notebook should be run top-to-bottom before committing.